### Connect the Google Drive to the notebook

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
os.chdir("/content/drive/MyDrive/dataset")

### Image path load

In [ ]:
import numpy as np
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.preprocessing import LabelEncoder
from PIL import Image
import pandas as pd
import cv2

In [ ]:
# Get the image paths and labels with in the input folder
def get_img_path_label(target_folder):
  img_path = []
  img_label = []
  for folder_name in os.listdir(target_folder):
      folder_path = os.path.join(target_folder, folder_name)
      if os.path.isdir(folder_path):
          for image_filename in os.listdir(folder_path):
              # Construct the full path to the image file
              image_path = os.path.join(folder_path, image_filename)
              # Append the image path to the lists
              img_path.append(image_path)
              # Append the image label to the lists
              img_label.append(folder_name)
  return img_path, img_label

# Get the image resolutions and sizes of given path images
def get_img_resolutions_sizes(img_paths):
  resolutions = []
  sizes = []
  # Loop over image paths and collect data
  for img_path in img_paths:
      # Read the image using OpenCV
      img = cv2.imread(img_path, cv2.IMREAD_UNCHANGED)
      # Get image resolution (height, width)
      height, width = img.shape[:2]
      resolutions.append((width, height))
      # Get image file size in bytes
      sizes.append(os.path.getsize(img_path))
  return resolutions, sizes

import glob
import os
import xml.etree.ElementTree as ET

def get_xml_paths(target_folder):
  xml_file_paths = []
  xml_files = glob.glob(os.path.join(target_folder, '*.xml'))
  # Now xml_files contains a list of full file paths to the .xml files
  for file_path in xml_files:
      xml_file_paths.append(file_path)
  return xml_file_paths

def parse_xml(xml_file):
    tree = ET.parse(xml_file)
    root = tree.getroot()

    image_info = {
        'filename': root.find('filename').text,
        'size': {
            'width': int(root.find('size/width').text),
            'height': int(root.find('size/height').text),
            'depth': int(root.find('size/depth').text),
        },
        'objects': [],
    }

    for obj in root.findall('object'):
        bndbox = obj.find('bndbox')
        object_info = {
            'name': obj.find('name').text,
            'bounding_box': {
                'xmin': int(bndbox.find('xmin').text),
                'ymin': int(bndbox.find('ymin').text),
                'xmax': int(bndbox.find('xmax').text),
                'ymax': int(bndbox.find('ymax').text),
            }
        }
        image_info['objects'].append(object_info)

    return image_info

def process_xml_files(xml_files):
    all_data = []
    for xml_file in xml_files:
        data = parse_xml(xml_file)
        all_data.append(data)
    return all_data

In [ ]:
pistol_classification = './Weapon/Pistol/Pistol_classification'
pistol_class_img_path, pistol_class_label = get_img_path_label(pistol_classification)

pistol_detection_xml = './Weapon/Pistol/Pistol_detection/xmls'
xml_files_p = get_xml_paths(pistol_detection_xml)
all_image_data_p = process_xml_files(xml_files_p)

pistol_dection_img = './Weapon/Pistol/Pistol_detection/Weapons'


knife_classification = './Weapon/Knife/Knife_classification'
# Get img paths and labels
knife_class_img_path, knife_class_label = get_img_path_label(knife_classification)


knife_detection_xml = './Weapon/Knife/Knife_detection/annotations'
xml_files_k = get_xml_paths(knife_detection_xml)
all_image_data_k = process_xml_files(xml_files_k)


knife_dection_img = './Weapon/Knife/Knife_detection/Images'

### Import

In [ ]:
from sklearn.model_selection import train_test_split
from torchvision import transforms, datasets
from torch.utils.data import DataLoader, Dataset
from sklearn.preprocessing import LabelEncoder
import torchvision.models as models
import torch
import os
from PIL import Image
from tqdm import tqdm

### Preparing the Data

Run Pistol or Knife for Data Preparation, then run the CNN model with desired variables

#### Pistol

In [ ]:
le = LabelEncoder()
# Convert labels to a binary labels: Pistol - 1; else - 0
binary_labels = [1 if label == 'AAAPistol' else 0 for label in pistol_class_label]
pistol_labels = le.fit_transform(binary_labels)

# Split the data into training and testing sets
train_img_paths, test_img_paths, train_labels, test_labels = train_test_split(
    pistol_class_img_path, pistol_labels, test_size=0.2, random_state=42
)

class CustomDataset(Dataset):
    def __init__(self, img_paths, labels, transform=None):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path)
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Define transformations, including a conversion to RGB
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# Create training and testing datasets and loaders
train_dataset = CustomDataset(train_img_paths, train_labels, transform=transform)
test_dataset = CustomDataset(test_img_paths, test_labels, transform=transform)

#### Knife

In [ ]:
le = LabelEncoder()
# Convert labels to a binary labels: Pistol - 1; else - 0
binary_labels = [1 if label == 'AAAKnife' else 0 for label in knife_class_label]
knife_labels = le.fit_transform(binary_labels)

# Split the data into training and testing sets
train_img_paths, test_img_paths, train_labels, test_labels = train_test_split(
    knife_class_img_path, knife_labels, test_size=0.2, random_state=42
)

class CustomDataset(Dataset):
    def __init__(self, img_paths, labels, transform=None):
        self.img_paths = img_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.img_paths)

    def __getitem__(self, idx):
        img_path = self.img_paths[idx]
        image = Image.open(img_path)
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Define transformations, including a conversion to RGB
transform = transforms.Compose([
    transforms.Lambda(lambda img: img.convert('RGB') if img.mode != 'RGB' else img),
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

# Create training and testing datasets and loaders
train_dataset = CustomDataset(train_img_paths, train_labels, transform=transform)
test_dataset = CustomDataset(test_img_paths, test_labels, transform=transform)

### CNN Model

#### Hyperparameters

In [ ]:
num_classes = 2
num_epochs = 10
learning_rate = 0.001
batch_size = 50

#### Define Dataloader

In [ ]:
# DataLoader with drop_last=True
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, drop_last=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, drop_last=True)

#### Define CNN Model

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 32 * 32, 512)
        self.fc2 = nn.Linear(512, num_classes)
        
    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(-1, 64 * 32 * 32)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

#### Train CNN Model

In [ ]:
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# num_classes = len(set(labels))  # Assuming labels are integer classes
pistol_class_model = SimpleCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(pistol_class_model.parameters(), lr=0.001)

# Training loop
for epoch in range(num_epochs):
    running_loss = 0.0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        outputs = pistol_class_model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
    print(f"Epoch {epoch+1}, Loss: {running_loss / len(train_loader)}")

#### Test CNN Model

In [ ]:
def test_model(loader, pistol_class_model):
    pistol_class_model.eval()  # Set the model to
    correct = 0
    total = 0
    with torch.no_grad():  # In testing phase, we don't need to compute gradients (for memory efficiency)
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = pistol_class_model(images)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total


In [ ]:
train_accuracy = test_model(train_loader, pistol_class_model)
test_accuracy = test_model(test_loader, pistol_class_model)
print(f'Training Accuracy: {train_accuracy:.2f}%')
print(f'Testing Accuracy: {test_accuracy:.2f}%')

### Save Model

In [ ]:
import pickle

# Save the model to a file
with open('knife_class_model_97.pkl', 'wb') as file:
    pickle.dump(pistol_class_model, file)